
# How priors push the dust posterior — flat vs narrow prior on τ_diff

If data is informative, the MAP estimate sits at the likelihood maximum
and prior choice barely matters. If data is uninformative, the MAP slides
toward the prior mode. At low S/N, the posterior shifts away from truth
when the prior is strong; at high S/N both priors converge.

Reference: Gelman+2013.


In [ ]:
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore")

ssp = tengri.load_ssp("fsps_prsc_miles_chabrier")
obs = tengri.Observation(
    photometry=tengri.Photometry.from_names(["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"])
)
model = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={
        "type": "dpl",
        "all_params": tengri.FIXED,
        "alpha": 1.5,
        "beta": 2.5,
        "tau_gyr": 0.5,
        "log_total_mass": 9.70,
    },
    dust={
        "type": "two_component",
        "all_params": tengri.FIXED,
        "tau_diff": tengri.Uniform(0.0, 2.0),
        "tau_bc": 0.2,
    },
    redshift=tengri.Fixed(0.05),
)

truth_tau = 0.5
params_truth = {
    **dict(model.spec.sample(jax.random.PRNGKey(0))),
    "dust_tau_diff": jnp.float64(truth_tau),
}
flux_truth = np.asarray(model.predict_photometry(params_truth))

tau_grid = np.linspace(0.02, 1.5, 120)


def log_lik(snr):
    sigma = flux_truth / snr
    rng = np.random.default_rng(42)
    f_obs = flux_truth + rng.normal(scale=sigma)
    ll = np.empty_like(tau_grid)
    for i, t in enumerate(tau_grid):
        p = {**params_truth, "dust_tau_diff": jnp.float64(t)}
        f_pred = np.asarray(model.predict_photometry(p))
        ll[i] = -0.5 * np.sum(((f_obs - f_pred) / sigma) ** 2)
    return ll - ll.max()


log_prior_flat = np.zeros_like(tau_grid)
log_prior_narrow = -0.5 * ((tau_grid - 0.3) / 0.1) ** 2

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
for ax, snr in zip(axes, [5.0, 50.0]):
    ll = log_lik(snr)
    post_flat = ll + log_prior_flat
    post_narrow = ll + log_prior_narrow
    post_flat -= post_flat.max()
    post_narrow -= post_narrow.max()

    ax.plot(tau_grid, np.exp(post_flat), color="C0", lw=1.6, label="flat prior")
    ax.plot(tau_grid, np.exp(post_narrow), color="C3", lw=1.6, label="narrow prior")
    ax.axvline(truth_tau, color="0.4", ls="--", lw=1.0)
    ax.text(truth_tau, 1.02, " truth", fontsize=9, color="0.4")
    ax.axvline(0.3, color="C3", ls=":", lw=0.8, alpha=0.5)
    ax.text(0.3, 0.92, " prior peak", fontsize=8, color="C3", alpha=0.7)
    ax.text(0.05, 0.93, rf"S/N = {snr:.0f}", transform=ax.transAxes, fontsize=10)
    ax.set(xlabel=r"$\tau_{\rm diff}$ (V-band)", ylim=(0, 1.1), xlim=(0, 1.5))
    if ax is axes[0]:
        ax.set_ylabel(r"posterior density (peak-normalized)")
    ax.legend(frameon=False, fontsize=9, loc="upper right")

fig.tight_layout()
plt.savefig("plot_prior_systematic_dust.png", dpi=150, bbox_inches="tight")